In [1]:
%load_ext watermark


In [2]:
import os
import subprocess

os.environ["POLARS_FORCE_NEW_STREAMING"] = "1"

import pandas as pd
import polars as pl
from tqdm import tqdm

from pylib._seed_global_rngs import seed_global_rngs


Covasim 3.1.6 (2024-01-28) — © 2020-2024 by IDM


In [3]:
pd.options.display.float_format = "{:,.1f}".format


In [4]:
%watermark -diwmuv -iv


Last updated: 2026-09-20T02:10:01.690835+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1064-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

pandas: 2.2.3
polars: 1.29.0

Watermark: 2.4.3



In [5]:
teeplot_subdir = "2025-05-30-compscreen-mutcount"
teeplot_subdir


'2025-05-30-compscreen-mutcount'

In [6]:
seed_global_rngs(1)


## Get Data


In [7]:
data_sources = {
    "uk": "https://osf.io/mkjy5/download",
    "multistrain": "https://osf.io/ywmpt/download",
    "vanilla": "https://osf.io/r8skg/download",
    "vanilla-big": "https://osf.io/j4795/download",
    "vanilla-big-1.3x": "https://osf.io/cnp5z/download",
}
tmp_path = f"/tmp/{teeplot_subdir}.pqt"


In [8]:
results = []

for source_name, url in data_sources.items():
    print(f"Downloading {source_name} data from {url}")

    subprocess.run(
        [
            "wget",
            "--tries=5",
            "--show-progress",
            "--progress=bar:force",
            "-O",
            str(tmp_path),
            url,
        ],
        check=True,
    )
    print("done!")

    df = pl.scan_parquet(
        tmp_path,
        low_memory=True,
        retries=5,
    )

    unique_groups = (
        df.unique(
            [
                "trt_name",
                "trt_n_downsample",
                "trt_hsurf_bits",
                "replicate_uuid",
            ]
        )
        .select(
            pl.col("trt_name"),
            pl.col("trt_n_downsample"),
            pl.col("trt_hsurf_bits"),
            pl.col("replicate_uuid"),
        )
        .drop_nans()
        .drop_nulls()
        .collect(engine="streaming")
    )

    for (trt_name, trt_n_downsample, trt_hsurf_bits, replicate_uuid) in tqdm(
        [*unique_groups.iter_rows()],
    ):
        group = df.filter(
            (pl.col("trt_name") == trt_name)
            & (pl.col("trt_n_downsample") == trt_n_downsample)
            & (pl.col("trt_hsurf_bits") == trt_hsurf_bits)
            & (pl.col("replicate_uuid") == replicate_uuid)
        ).collect(engine="streaming")

        group_df = group.to_pandas()
        res = [
            {
                "sum leaf count": group_df.loc[
                    group_df["is_focal_defmut"],
                    "num_leaves",
                ].sum(),
                "sum defmut": group_df["is_focal_defmut"].astype(bool).sum(),
            },
        ]
        results.extend(
            {
                "source_name": source_name,
                "trt_name": trt_name,
                "trt_n_downsample": trt_n_downsample,
                "trt_hsurf_bits": trt_hsurf_bits,
                "replicate_uuid": replicate_uuid,
                **record,
            }
            for record in res
        )


--2026-09-20 02:10:01--  https://osf.io/mkjy5/download
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://osf.io/download/mkjy5 [following]
--2026-09-20 02:10:02--  https://osf.io/download/mkjy5
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 308 Permanent Redirect
Location: https://osf.io/download/mkjy5/ [following]
--2026-09-20 02:10:02--  https://osf.io/download/mkjy5/
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 302 Found
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682c8b8ca9789d8b628f3f38 [following]
--2026-09-20 02:10:02--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682c8b8ca9789d8b628f3f38
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP reques

done!


100%|██████████| 70/70 [00:28<00:00,  2.46it/s]
--2026-09-20 02:10:33--  https://osf.io/ywmpt/download
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://osf.io/download/ywmpt [following]
--2026-09-20 02:10:33--  https://osf.io/download/ywmpt
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 308 Permanent Redirect
Location: https://osf.io/download/ywmpt/ [following]
--2026-09-20 02:10:33--  https://osf.io/download/ywmpt/
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 

302 Found
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682cae3abb3f815770b06cc8 [following]
--2026-09-20 02:10:33--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682cae3abb3f815770b06cc8
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://storage.googleapis.com/cos-osf-prod-files-us-east1/b6c38b1982ac61dc8727c85a0d9202e70b0b6f96a8465ecc658c156a647fc1dc?response-content-disposition=attachment%3B%20filename%3D%22a%3Dresult%2Bdate%3D2025-05-20%2Bjob%3D2025-05-17-multistrain-run_compscreen%2Bext%3D.pqt%22%3B%20filename%2A%3DUTF-8%27%27a%253Dresult%252Bdate%253D2025-05-20%252Bjob%253D2025-05-17-multistrain-run_compscreen%252Bext%253D.pqt&GoogleAccessId=files-us%40cos-osf-prod.iam.gserviceaccount.com&Expires=1789870294&Signature=KsTRaJ1H5RRlqVuENrNVS%2B4BR4%2Bd5oVkNXWxfJZXNzLbV%2B5TqZDpV2UcPZxdmONRdRC32hX

done!


...

done!


100%|██████████| 18/18 [00:14<00:00,  1.28it/s]
--2026-09-20 02:13:20--  https://osf.io/cnp5z/download
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://osf.io/download/cnp5z [following]
--2026-09-20 02:13:20--  https://osf.io/download/cnp5z
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 

308 Permanent Redirect
Location: https://osf.io/download/cnp5z/ [following]
--2026-09-20 02:13:20--  https://osf.io/download/cnp5z/
Reusing existing connection to osf.io:443.
HTTP request sent, awaiting response... 302 Found
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682caa9fbb3f815770b06c6b [following]
--2026-09-20 02:13:20--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682caa9fbb3f815770b06c6b
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://storage.googleapis.com/cos-osf-prod-files-us-east1/2699685afb8432314e32aeb42461c6e793a5657919f106fac95e2c3836dcd309?response-content-disposition=attachment%3B%20filename%3D%22a%3Dresult%2Bdate%3D2025-05-20%2Bjob%3D2025-05-18-vanilla-big-1-3x-run_compscreen%2Bext%3D.pqt%22%3B%20filename%2A%3DUTF-8%27%27a%253Dresult%252Bdate%253D2025-05-20%252Bjob%253D2025

done!


100%|██████████| 20/20 [00:22<00:00,  1.11s/it]


In [9]:
results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{teeplot_subdir}-raw.csv",
    index=False,
)
results_df


,source_name,trt_name,trt_n_downsample,trt_hsurf_bits,replicate_uuid,sum leaf count,sum defmut
0,uk,Sben2x/Gdel2x,1000000,0,0f07d386-bc29-8b1c-9c27-adc93df6f99f,522,274
1,uk,Sben1.1x/Gneu,1000000,64,1295ccfb-605c-88c9-8ebf-900370e800d0,1,1
2,uk,Sben2x/Gneu,1000000,0,29143eaa-000f-832e-95e4-edb14e6958b1,9508,600
3,uk,Sben2x/Gneu,1000000,64,29143eaa-000f-832e-95e4-edb14e6958b1,398,398
4,uk,Sben1.3x/Gdel1.3x,1000000,64,6cca2f5d-7303-80aa-9b56-c65a26f04eba,9,9
...,...,...,...,...,...,...,...
172,vanilla-big-1.3x,Sben1.3x/Gdel1.3x,200000,0,c81cab91-5dfe-83d3-a088-110240579728,420,237
173,vanilla-big-1.3x,Sben1.3x/Gdel1.3x,2000000,0,2b2da60e-53a7-8888-9058-ce1b89fb83e3,4805,1298
174,vanilla-big-1.3x,Sben1.3x/Gdel1.3x,200000,0,15f36e76-1fe6-861d-bf0c-b171c07713df,371,242
175,vanilla-big-1.3x,Sben1.3x/Gdel1.3x,200000,0,2b2da60e-53a7-8888-9058-ce1b89fb83e3,485,272


In [10]:
summary_df = results_df.groupby(
    ["source_name", "trt_name", "trt_n_downsample", "trt_hsurf_bits"]
).agg(
    {
        "sum leaf count": ["mean", "std"],
        "sum defmut": ["mean", "std"],
    },
)
summary_df.to_csv(
    f"{teeplot_subdir}-summary.csv",
    index=True,
)
summary_df


sum leaf count  \
                                                                             mean   
source_name      trt_name          trt_n_downsample trt_hsurf_bits                  
multistrain      Sben1.1x/Gdel1.1x 1000000          0                       865.8   
                 Sben1.1x/Gneu     1000000          0                       323.4   
                 Sben1.3x/Gdel1.3x 1000000          0                     1,543.8   
                 Sben1.3x/Gneu     1000000          0                     6,386.8   
                 Sben2x/Gdel2x     1000000          0                    12,510.8   
                 Sben2x/Gneu       1000000          0                   105,072.2   
                 Sneu/Gneu         1000000          0                        92.0   
uk               Sben1.1x/Gdel1.1x 1000000          0                         1.6   
                                                    64                        1.2   
                 Sben1.1x/Gneu     1000000          0                         3.2   
                                                    64                        1.0   
                 Sben1.3x/Gdel1.3x 1000000          0                        67.8   
                                                    64                        9.0   
                 Sben1.3x/Gneu     1000000          0                       130.0   
                                                    64                       10.4   
                 Sben2x/Gdel2x     1000000          0                       559.2   
                                                    64                      210.2   
                 Sben2x/Gneu       1000000          0                     6,436.6   
                                                    64                      348.6   
                 Sneu/Gneu         1000000          0                         1.2   
                                                    64                        0.6   
vanilla          Sben1.1x/Gdel1.1x 1000000          0                       141.2   
                 Sben1.1x/Gneu     1000000          0                       525.0   
                 Sben1.3x/Gdel1.3x 1000000          0                     6,154.4   
                 Sben1.3x/Gneu     1000000          0                    29,039.6   
                 Sben2x/Gdel2x     1000000          0                    18,099.4   
                 Sben2x/Gneu       1000000          0                   299,979.0   
                 Sneu/Gneu         1000000          0                        85.0   
vanilla-big      Sben1.1x/Gdel1.1x 200000           0                       201.0   
                                                    64                       12.7   
                                   2000000          0                     2,684.7   
                 Sben1.1x/Gneu     200000           0                       356.0   
                                                    64                       30.0   
                                   2000000          0                     4,005.7   
vanilla-big-1.3x Sben1.3x/Gdel1.3x 200000           0                       409.2   
                                   2000000          0                     4,892.6   
                 Sben1.3x/Gneu     200000           0                     5,819.0   
                                   2000000          0                    58,494.4   

                                                                             \
                                                                        std   
source_name      trt_name          trt_n_downsample trt_hsurf_bits            
multistrain      Sben1.1x/Gdel1.1x 1000000          0                 438.4   
                 Sben1.1x/Gneu     1000000          0                 241.1   
                 Sben1.3x/Gdel1.3x 1000000          0                 380.3   
                 Sben1.3x/Gneu     1000000          0               2,054.2   
                 Sben2x/Gdel